# 🦾 Fine-Tuning SmolVLA on SO-100 Robot Arm

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chalee88/SmolVLA/blob/main/notebooks/smolvla_so100_finetuning.ipynb)

This notebook demonstrates how to fine-tune **[SmolVLA](https://huggingface.co/papers/2506.01844)** (Hugging Face's 450M Vision-Language-Action foundation model) on the **[SO-100 Pick & Place Dataset](https://huggingface.co/datasets/lerobot/svla_so100_pickplace)** using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

### 📋 What this notebook covers:
1. **Environment Setup**: Installing LeRobot and verifying CUDA GPU availability.
2. **Dataset Inspection**: Loading and visualizing the dual-camera views (Top & Wrist) and motor states.
3. **Base Model Verification**: Running a pre-training forward pass with `lerobot/smolvla_base`.
4. **Fine-Tuning**: Training the action expert on 50 pick-and-place demonstration episodes.
5. **Inference & Evaluation**: Generating continuous 6-DoF joint actions with the fine-tuned checkpoint.
6. **Export**: Zipping and downloading your fine-tuned model weights.

## 1. Verify GPU & Install Dependencies
Ensure you are using a GPU runtime (**Runtime -> Change runtime type -> T4 GPU**).

In [ ]:
# Check GPU availability
!nvidia-smi

# Clone LeRobot and install dependencies for training and SmolVLA
!git clone https://github.com/huggingface/lerobot.git
!apt-get update && apt-get install -y ffmpeg
!cd lerobot && pip install -q -e ".[dataset,training]"
!pip install -q matplotlib

## 2. Load & Inspect Dataset
We load the official `lerobot/svla_so100_pickplace` dataset collected on physical SO-100 hardware.

In [ ]:
import sys
sys.path.append("lerobot/src")

from lerobot.datasets.lerobot_dataset import LeRobotDataset
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt

repo_id = "lerobot/svla_so100_pickplace"
print(f"Downloading and loading dataset: {repo_id}...")
dataset = LeRobotDataset(repo_id)

print(f"✓ Total frames: {len(dataset)}")
print(f"✓ Total episodes: {dataset.num_episodes}")

# Extract a sample
sample = dataset[0]
img_top = TF.to_pil_image(sample["observation.images.top"])
img_wrist = TF.to_pil_image(sample["observation.images.wrist"])

# Plot camera perspectives
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_top)
axes[0].set_title(f"Top Camera\nTask: {sample['task']}")
axes[0].axis("off")

axes[1].imshow(img_wrist)
axes[1].set_title("Wrist (Gripper) Camera")
axes[1].axis("off")
plt.tight_layout()
plt.show()

print("Initial Motor Joint Angles:", [round(x, 2) for x in sample["observation.state"].tolist()])

## 3. Fine-Tune SmolVLA on the Dataset

We fine-tune `lerobot/smolvla_base` using `lerobot-train`.

> **Important Config Details:**
> - `--rename_map`: Maps the dataset cameras (`top`, `wrist`) to SmolVLA's expected inputs (`camera1`, `camera2`).
> - `--batch_size=8`: Fits comfortably in Colab's 15GB T4 GPU VRAM.
> - `--policy.freeze_vision_encoder=true`: Keeps the SmolVLM vision backbone frozen to save VRAM and train only the action expert.

In [ ]:
!cd lerobot && python -m lerobot.scripts.lerobot_train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=lerobot/svla_so100_pickplace \
  --rename_map='{"observation.images.top": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}' \
  --batch_size=8 \
  --steps=10000 \
  --save_freq=2500 \
  --output_dir=outputs/train/my_smolvla_so100 \
  --job_name=smolvla_so100_run \
  --policy.device=cuda \
  --wandb.enable=false \
  --policy.push_to_hub=false

## 4. Test Action Prediction with Fine-Tuned Model
Load the fine-tuned checkpoint and verify predicted action chunks for the SO-100 robot arm.

In [ ]:
import torch
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.smolvla.processor_smolvla import make_smolvla_pre_post_processors

checkpoint_path = "lerobot/outputs/train/my_smolvla_so100/checkpoints/last/pretrained_model"
print(f"Loading checkpoint from: {checkpoint_path}...")

# Load trained policy
policy = SmolVLAPolicy.from_pretrained(checkpoint_path)
policy.to("cuda")
policy.eval()

# Configure processor with rename map
preprocessor, postprocessor = make_smolvla_pre_post_processors(
    config=policy.config,
    dataset_stats=dataset.meta.stats,
)

rename_map = {
    "observation.images.top": "observation.images.camera1",
    "observation.images.wrist": "observation.images.camera2",
}
for step in preprocessor.steps:
    if hasattr(step, "rename_map"):
        step.rename_map = rename_map

# Preprocess sample
inputs = preprocessor(dataset[0])

# Predict action chunk
with torch.no_grad():
    predicted_actions = policy.select_action(inputs)
    predicted_actions = postprocessor(predicted_actions)

print("✓ Action chunk predicted successfully!")
print("Action chunk shape:", predicted_actions.shape)
print("First 3 action steps (degrees):")
print(predicted_actions[:3])

## 5. Download the Fine-Tuned Model
Zip the final checkpoint folder so you can download it to your PC and deploy it on your robot!

In [ ]:
import shutil
from google.colab import files

# Create zip archive of the trained weights
archive_name = "my_smolvla_so100_checkpoint"
shutil.make_archive(archive_name, "zip", checkpoint_path)

print("Downloading fine-tuned weights to your PC...")
files.download(f"{archive_name}.zip")